<a href="https://colab.research.google.com/github/Chosencodes/Cardiac_Heart_Detection/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/cardiac-heart-detection", exist_ok=True)

In [ ]:
!pip install pytorch-lightning torchmetrics


In [ ]:
import torch
import torchvision
import numpy as np
import pandas as pd
import albumentations as A
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

In [ ]:
labels_path   = "/content/drive/MyDrive/05-Detection/rsna_heart_detection.csv"
train_patients = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/train_subjects.npy"
val_patients   = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/val_subjects.npy"
train_root     = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/train/"
val_root       = "/content/drive/MyDrive/cardiac-heart-detection/Processed-Heart-Detection/val/"

In [ ]:
train_augs = A.Compose([
    A.RandomGamma(p=0.5),
    A.Affine(scale = (0.8, 1.2),rotate = (-10, 10),translate_px = {"x": (-10, 10), "y": (-10, 10)}),
], bbox_params=A.BboxParams(format = "pascal_voc",label_fields = ["labels"], clip = True
))


In [ ]:
dataset_code = '''
import torch
import numpy as np
import pandas as pd
import albumentations as A
from pathlib import Path
import os

class CardiacDataset(torch.utils.data.Dataset):
  def __init__(self, path_to_labels_csv, patients, root_path, augs=None):
    self.labels = pd.read_csv(path_to_labels_csv)
    self.patients = np.load(patients)
    self.root_path = Path(root_path)
    self.augment = augs

  def __len__(self):
    return len(self.patients)

  def __getitem__(self, idx):
    patient = self.patients[idx]
    data = self.labels[self.labels["name"] == patient]

    x_min = data["x0"].item()
    y_min = data["y0"].item()
    x_max = x_min + data["w"].item()
    y_max = y_min + data["h"].item()

    file_path = self.root_path / patient / f"{patient}.npy"
    img = np.load(str(file_path)).astype(np.float32)

    if self.augment:
        img_uint8 = (img * 255).clip(0, 255).astype(np.uint8)
        img_uint8 = np.expand_dims(img_uint8, axis=-1)

        transformed = self.augment(
            image=img_uint8,
            bboxes=[[x_min, y_min, x_max, y_max]],
            labels=["heart"]
        )

        img = transformed["image"].squeeze(-1).astype(np.float32) / 255.0
        if transformed["bboxes"]:
            x_min, y_min, x_max, y_max = transformed["bboxes"][0]

    img = (img - 0.494) / 0.253
    img = torch.tensor(img).unsqueeze(0)
    bbox = torch.tensor([x_min, y_min, x_max, y_max])

    return img, bbox
'''

dataset_code = """
import torch
import numpy as np
import pandas as pd
from pathlib import Path

class CardiacDataset(torch.utils.data.Dataset):

    def __init__(self, path_to_labels_csv, patients, root_path, augs=None):
        self.labels = pd.read_csv(path_to_labels_csv)
        self.patients = np.load(patients)
        self.root_path = Path(root_path)
        self.augment = augs

    def __len__(self):
        return len(self.patients)

    def __getitem__(self, idx):
        patient = self.patients[idx]
        data = self.labels[self.labels["name"] == patient]

        x_min = data["x0"].item()
        y_min = data["y0"].item()
        x_max = x_min + data["w"].item()
        y_max = y_min + data["h"].item()

        file_path = self.root_path / patient / f"{patient}.npy"
        img = np.load(str(file_path)).astype(np.float32)

        img = torch.tensor(img).unsqueeze(0)
        bbox = torch.tensor([x_min, y_min, x_max, y_max])

        return img, bbox
"""

with open("/content/drive/MyDrive/cardiac-heart-detection/dataset.py", "w") as f:
    f.write(dataset_code)

import sys

sys.path.append("/content/drive/MyDrive/cardiac-heart-detection")

import importlib
import dataset
importlib.reload(dataset)

from dataset import CardiacDataset

In [ ]:
import sys
if "/content" not in sys.path:
    sys.path.append("/content")

import importlib
import dataset
importlib.reload(dataset)

from dataset import CardiacDataset

In [ ]:
train_dataset = CardiacDataset(labels_path, train_patients, train_root, augs=train_augs)
val_dataset   = CardiacDataset(labels_path, val_patients, val_root, augs=None)

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset,batch_size=8,num_workers=4,shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset,batch_size=8,num_workers=4,shuffle=False)

In [ ]:
print(f"There are {len(train_dataset)} train images and {len(val_dataset)} val images")

In [ ]:
# torchvision.models.resnet50()

In [ ]:
class CardiacDetectionModel(pl.LightningModule):
  def __init__(self):
    super().__init__()
    self.model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
    self.model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    self.model.fc = torch.nn.Linear(in_features=2048, out_features=4)

    self.loss_fn = torch.nn.MSELoss()

  def forward(self,data):
    return self.model(data)

  def training_step(self,batch,batch_idx):
    x_ray,label = batch
    label = label.float()
    pred = self(x_ray)
    loss = self.loss_fn(pred,label)

    self.log("Train loss",loss,prog_bar=True)

    if batch_idx % 50 == 0:
      self.log_images(x_ray.cpu(),pred.cpu(),label.cpu(),"Train")
    return loss

  def validation_step(self,batch,batch_idx):
    x_ray,label = batch
    label = label.float()
    pred = self(x_ray)
    loss = self.loss_fn(pred,label)

    self.log("Val loss",loss,prog_bar=True)

    if batch_idx % 50 == 0:
      self.log_images(x_ray.cpu(),pred.cpu(),label.cpu(),"Val")
    return loss

  def log_images(self,x_ray,pred,label,name):
    results = []

    for i in range(4):
      coords_labels = label[i]
      coords_pred = pred[i]

      img = ((x_ray[i] * 0.253) + 0.494).numpy().squeeze()
      img = np.stack([img, img, img], axis=-1).copy()

      x0,y0 = coords_labels[0].int().item(),coords_labels[1].int().item()
      x1,y1 = coords_labels[2].int().item(),coords_labels[3].int().item()
      img = cv2.rectangle(img,(x0,y0),(x1,y1),(0,0,0),2)

      x0,y0 = coords_pred[0].int().item(),coords_pred[1].int().item()
      x1,y1 = coords_pred[2].int().item(),coords_pred[3].int().item()
      img = cv2.rectangle(img,(x0,y0),(x1,y1),(1,1,1),2)

      results.append(torch.tensor(img).permute(2, 0, 1))

    grid = torchvision.utils.make_grid(results,nrow=2)
    self.logger.experiment.add_image(name,grid,self.global_step)

  def configure_optimizers(self):
      optimizer = torch.optim.Adam(self.parameters(),lr=1e-4)
      return optimizer

In [ ]:
model = CardiacDetectionModel()
print(hasattr(model, 'configure_optimizers'))

In [ ]:
checkpoint_callback = ModelCheckpoint(
    monitor="Val loss",
    save_top_k=10,
    mode="min",
    filename="best-checkpoint"
)

In [ ]:
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    max_epochs=152,
    callbacks=[checkpoint_callback],
    logger = TensorBoardLogger(save_dir="/content/logs"),
    log_every_n_steps=1
)

In [ ]:
trainer.fit(model,train_loader,val_loader)

# **Evaluation**

In [ ]:
best_model_path = checkpoint_callback.best_model_path
print(f"Best checkpoint: {best_model_path}")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = CardiacDetectionModel.load_from_checkpoint(
    best_model_path,
    map_location=device
)
model = model.to(device)
model.eval()
print("Model loaded successfully!")

In [ ]:
import shutil

best_model_path = checkpoint_callback.best_model_path

dest = "/content/drive/MyDrive/cardiac-heart-detection/best-checkpoint.ckpt"

shutil.copy(best_model_path, dest)

import os
print("✅ Saved" if os.path.exists(dest) else "❌ Failed")
print("Size (MB):", os.path.getsize(dest) / 1e6)

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

model = CardiacDetectionModel.load_from_checkpoint(
    "/content/drive/MyDrive/cardiac-heart-detection/best-checkpoint.ckpt",
    map_location=device
)
model = model.to(device)
model.eval()
print("Model loaded successfully!")
print(f"Best checkpoint: {best_model_path}")

In [ ]:
preds = []
labels = []

with torch.no_grad():
  for data,label in val_dataset:
    data = data.to(device).float().unsqueeze(0)
    pred = model(data)[0].cpu()
    preds.append(pred)
    labels.append(label)

preds=torch.stack(preds)
labels=torch.stack(labels)

In [ ]:
IDX = 12
img, label = val_dataset[IDX]
pred = preds[IDX]

fig,axis = plt.subplots(1,1)
axis.imshow(img[0],cmap="bone")
pred_box = patches.Rectangle((pred[0],pred[1]),pred[2]-pred[0],pred[3]-pred[1],edgecolor="r",facecolor="none")

label_box = patches.Rectangle((label[0], label[1]), label[2]-label[0], label[3]-label[1],
                                edgecolor="g", facecolor="none", label="Ground Truth")

axis.add_patch(pred_box)
axis.add_patch(label_box)
axis.legend()
plt.title(f"Sample {IDX} — Red: Prediction | Green: Ground Truth")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 12))
c = 0
for i in range(2):
    for j in range(3):
        img, label = val_dataset[c]
        pred = preds[c]

        axes[i][j].imshow(img[0], cmap="bone")

        pred_box = patches.Rectangle((pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
                                      edgecolor="r", facecolor="none")
        label_box = patches.Rectangle((label[0], label[1]), label[2]-label[0], label[3]-label[1],
                                       edgecolor="g", facecolor="none")
        axes[i][j].add_patch(pred_box)
        axes[i][j].add_patch(label_box)
        axes[i][j].set_title(f"IDX {c}")
        c += 1
plt.tight_layout()
plt.show()

In [ ]:
abs(preds-labels).mean(0)

In [ ]:
import torchmetrics
import torch

preds_tensor = torch.stack(preds) if isinstance(preds, list) else preds
labels_tensor = torch.stack([val_dataset[i][1] for i in range(len(val_dataset))])

mae = torch.abs(preds_tensor - labels_tensor).mean(0)
print(f"MAE [x_min, y_min, x_max, y_max]: {mae}")


print(f"Overall MAE: {mae.mean():.4f} pixels")

iou_scores = []
for pred, label in zip(preds_tensor, labels_tensor):

    x1 = torch.max(pred[0], label[0])
    y1 = torch.max(pred[1], label[1])
    x2 = torch.min(pred[2], label[2])
    y2 = torch.min(pred[3], label[3])

    intersection = torch.clamp(x2-x1, min=0) * torch.clamp(y2-y1, min=0)

    pred_area  = (pred[2]-pred[0])  * (pred[3]-pred[1])
    label_area = (label[2]-label[0]) * (label[3]-label[1])
    union = pred_area + label_area - intersection

    iou_scores.append(intersection / union)

iou = torch.stack(iou_scores).mean()
print(f"Mean IoU: {iou:.4f}")
print(f"IoU as %: {iou*100:.1f}%")

In [ ]:
import os
os.makedirs("/content/drive/MyDrive/cardiac-heart-detection/results", exist_ok=True)


iou_scores = []
for pred, label in zip(preds, labels):
    x1 = torch.max(pred[0], label[0])
    y1 = torch.max(pred[1], label[1])
    x2 = torch.min(pred[2], label[2])
    y2 = torch.min(pred[3], label[3])

    intersection = torch.clamp(x2-x1, min=0) * torch.clamp(y2-y1, min=0)
    pred_area    = (pred[2]-pred[0])  * (pred[3]-pred[1])
    label_area   = (label[2]-label[0]) * (label[3]-label[1])
    union        = pred_area + label_area - intersection
    iou_scores.append((intersection / union).item())

iou_scores = torch.tensor(iou_scores)

print("=== CONFIDENCE SCORES ===")
print(f"Mean confidence:   {iou_scores.mean()*100:.1f}%")
print(f"High confidence (IoU>0.8): {(iou_scores > 0.8).sum().item()} / {len(iou_scores)} samples")
print(f"Low confidence  (IoU<0.5): {(iou_scores < 0.5).sum().item()} / {len(iou_scores)} samples")

print("\n=== HEART SIZE MEASUREMENTS ===")
heart_widths  = preds[:, 2] - preds[:, 0]
heart_heights = preds[:, 3] - preds[:, 1]

print(f"Avg predicted heart width:  {heart_widths.mean():.1f} pixels")
print(f"Avg predicted heart height: {heart_heights.mean():.1f} pixels")

enlarged = 0
for i, (w, h) in enumerate(zip(heart_widths, heart_heights)):
    if w > 112:
        enlarged += 1

print(f"Flagged as enlarged (cardiomegaly): {enlarged} / {len(preds)} samples")

print("\n=== CARDIOTHORACIC RATIO ===")
image_width = 224
ct_ratios   = heart_widths / image_width

print(f"Mean CT ratio: {ct_ratios.mean():.3f}")
print(f"Normal   (CT ratio ≤ 0.5): {(ct_ratios <= 0.5).sum().item()} / {len(preds)} samples")
print(f"Abnormal (CT ratio > 0.5): {(ct_ratios > 0.5).sum().item()} / {len(preds)} samples")

IDX = 5
img, label = val_dataset[IDX]
pred        = preds[IDX]
ct_ratio    = (pred[2] - pred[0]) / image_width
confidence  = iou_scores[IDX] * 100
status = "Enlarged" if ct_ratio > 0.5 else "Normal"

fig, axis = plt.subplots(1, 1, figsize=(6, 6))
axis.imshow(img[0], cmap="bone")

pred_box  = patches.Rectangle((pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
                                edgecolor="r", facecolor="none", label=f"Pred (conf: {confidence:.1f}%)")
label_box = patches.Rectangle((label[0], label[1]), label[2]-label[0], label[3]-label[1],
                                edgecolor="g", facecolor="none", label="Ground Truth")
axis.add_patch(pred_box)
axis.add_patch(label_box)
axis.legend()
axis.set_title(f"CT Ratio: {ct_ratio:.2f} | {status} | Confidence: {confidence:.1f}%")
plt.savefig("/content/drive/MyDrive/cardiac-heart-detection/results/sample_predictions.png")
plt.show()
print("Saved!")

# **Gradio App**

In [ ]:
app_code = '''
import torch
import torchvision
import pytorch_lightning as pl
import numpy as np
import cv2
import gradio as gr
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

class CardiacDetectionModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.model = torchvision.models.resnet50(weights=None)
        self.model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7,7), stride=(2,2), padding=(3,3), bias=False)
        self.model.fc = torch.nn.Linear(in_features=2048, out_features=4)
        self.loss_fn = torch.nn.MSELoss()

    def forward(self, data):
        return self.model(data)

    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=1e-4)

device = torch.device("cpu")
model = CardiacDetectionModel.load_from_checkpoint(
    "best-checkpoint.ckpt",
    map_location=device
)
model.eval()

def detect_heart(image):
    img = np.array(Image.fromarray(image).convert("L"))
    img = cv2.resize(img, (224, 224)).astype(np.float32)
    img = img / 255.0
    img = (img - 0.494) / 0.253

    tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).float()
    with torch.no_grad():
        pred = model(tensor)[0].cpu()

    heart_width = (pred[2] - pred[0]).item()
    ct_ratio    = heart_width / 224
    status      = "Enlarged (Possible Cardiomegaly)" if ct_ratio > 0.5 else "Normal"

    fig, axis = plt.subplots(1, 1, figsize=(6, 6))
    axis.imshow(img, cmap="bone")
    pred_box = patches.Rectangle(
        (pred[0], pred[1]), pred[2]-pred[0], pred[3]-pred[1],
        edgecolor="r", facecolor="none", linewidth=2, label="Heart Detection"
    )
    axis.add_patch(pred_box)
    axis.legend()
    axis.set_title(f"CT Ratio: {ct_ratio:.2f} | {status}")
    axis.axis("off")
    fig.savefig("output.png", bbox_inches="tight")
    plt.close()
    return "output.png", f"CT Ratio: {ct_ratio:.3f}", status

demo = gr.Interface(
    fn=detect_heart,
    inputs=gr.Image(label="Upload Chest X-Ray"),
    outputs=[
        gr.Image(label="Detection Result"),
        gr.Text(label="Cardiothoracic Ratio"),
        gr.Text(label="Heart Status"),
    ],
    title="Cardiac Heart Detection",
    description="Upload a frontal chest X-ray to detect heart location and calculate cardiothoracic ratio.",
)
demo.launch()
'''

with open("/content/drive/MyDrive/cardiac-heart-detection/app.py", "w") as f:
    f.write(app_code)
print("app.py created!")

In [ ]:
requirements = """torch
torchvision
gradio
opencv-python
numpy
Pillow
pytorch-lightning
"""

with open("/content/drive/MyDrive/cardiac-heart-detection/requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt created!")

In [ ]:

IDX = 10
img, label = val_dataset[IDX]

img_display = (img[0].numpy() * 0.253) + 0.494
img_display = np.clip(img_display, 0, 1)

import matplotlib.pyplot as plt
plt.imsave(
    "/content/drive/MyDrive/cardiac-heart-detection/test_xray.png",
    img_display,
    cmap="gray",
    vmin=0,
    vmax=1
)
print("Saved! Download from Drive and upload to the app.")